In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import torch


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
data_folder = 'data'

# 根据周期标签与伪衰老得分加载数据
def load_cycle_data(file_name, cycle_label, pseudo_age_score):
    # 读取数据
    df = pd.read_csv(os.path.join(data_folder, file_name))
    
    # 删除Num.中无关行
    df = df[~df['Num.'].str.contains('平均值', case=False, na=False)] 
    df = df[~df['Num.'].str.contains('AVE.', case=False, na=False)] 
    # 假设平均值行中有 cell_id 字段包含'average'
    
    # 添加周期与伪衰老得分
    df['cycle'] = cycle_label
    df['aging_score'] = pseudo_age_score
    
    return df


In [4]:
# 加载并合并所有周期数据
# 设定伪标签
data_files = [
    ('UCSC_P4.csv', 'P4', 0.0),
    ('UCSC_P6.csv', 'P6', 0.33),
    ('UCSC_P8.csv', 'P8', 0.67),
    ('UCSC_P10.csv', 'P10', 1.0)
]

In [5]:
df_list = [load_cycle_data(f, label, score) for f, label, score in data_files]
df_all = pd.concat(df_list, ignore_index=True)


df_all = df_all.rename(columns={
    'Height（ um）': 'Height',
    'Roughness （nm）':'Roughness',
    'Adhesion (pN)':'Adhesion'
          
})   

df_all.columns = df_all.columns.str.strip()  #去除列名中的空格
df_all.columns = df_all.columns.str.replace(' ','_')  #使用_取代空格
df_all.columns = df_all.columns.str.lower() #全小写的 

df_all.items

<bound method DataFrame.items of       unnamed:_0 num.  height  roughness  length  wideth  adhesion  \
0            NaN    1   3.429      78.26   69.68   28.48     452.9   
1            NaN  NaN   3.372      84.80   69.81   28.43     452.5   
2            NaN  NaN   3.297     112.70   69.46   27.23     454.8   
3            NaN  NaN   3.398      90.45   69.96   26.94     453.2   
4            NaN  NaN   3.391      76.28   70.15   26.08     456.3   
...          ...  ...     ...        ...     ...     ...       ...   
7995         NaN  NaN   1.953     169.74   95.06   38.42     612.4   
7996         NaN  NaN   1.947     174.30   94.96   38.15     613.7   
7997         NaN  NaN   1.926     178.05   95.50   38.54     618.3   
7998         NaN  NaN   1.909     190.50   95.20   38.52     618.8   
7999         NaN  NaN   1.933     182.00   95.27   38.29     623.2   

      elastic_modulus  aspect_ratio cycle  aging_score  unnamed:_9  \
0               3.294      2.446629    P4          0.0  

In [6]:
Featrue_column = ['height','roughness','length','wideth','adhesion','elastic_modulus','cycle','aging_score']
df_all = df_all[[col for col in df_all.columns if col in Featrue_column]]
if 'cell_id' not in df_all.columns:
    # 假设数据每细胞20条连续记录（按文件原始顺序）
    df_all = df_all.reset_index(drop=True)
    df_all['cell_id'] = df_all.index // 20
    print("已生成 cell_id（假设每细胞20条观测）。")

df_all.head(10)

已生成 cell_id（假设每细胞20条观测）。


,height,roughness,length,wideth,adhesion,elastic_modulus,cycle,aging_score,cell_id
0,3.429,78.26,69.68,28.48,452.9,3.294,P4,0.0,0
1,3.372,84.80,69.81,28.43,452.5,2.453,P4,0.0,0
2,3.297,112.70,69.46,27.23,454.8,3.330,P4,0.0,0
3,3.398,90.45,69.96,26.94,453.2,3.350,P4,0.0,0
4,3.391,76.28,70.15,26.08,456.3,3.040,P4,0.0,0
5,3.430,77.91,69.72,26.74,454.9,3.740,P4,0.0,0
6,3.416,84.80,69.84,27.07,453.9,3.040,P4,0.0,0
7,3.410,121.10,70.26,27.96,456.7,3.000,P4,0.0,0
8,3.341,92.92,70.04,27.66,452.5,2.770,P4,0.0,0
9,3.295,78.63,69.58,26.29,453.9,3.080,P4,0.0,0


In [7]:
csv_filename = f"all.csv"
csv_path = os.path.join(data_folder, csv_filename)
# 保存为CSV

df_all.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✅ 已转换子表: all.csv")

✅ 已转换子表: all.csv


In [8]:
df_all.items

<bound method DataFrame.items of       height  roughness  length  wideth  adhesion  elastic_modulus cycle  \
0      3.429      78.26   69.68   28.48     452.9            3.294    P4   
1      3.372      84.80   69.81   28.43     452.5            2.453    P4   
2      3.297     112.70   69.46   27.23     454.8            3.330    P4   
3      3.398      90.45   69.96   26.94     453.2            3.350    P4   
4      3.391      76.28   70.15   26.08     456.3            3.040    P4   
...      ...        ...     ...     ...       ...              ...   ...   
7995   1.953     169.74   95.06   38.42     612.4            7.180   P10   
7996   1.947     174.30   94.96   38.15     613.7            7.158   P10   
7997   1.926     178.05   95.50   38.54     618.3            8.057   P10   
7998   1.909     190.50   95.20   38.52     618.8            8.033   P10   
7999   1.933     182.00   95.27   38.29     623.2            8.137   P10   

      aging_score  cell_id  
0             0.0        

In [47]:
# 随机打乱数据
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

In [48]:
# 特征与目标变量
features = ['height', 'roughness', 'length', 'wideth', 'adhesion', 'elastic_modulus']
X = df_all[features].values
y = df_all['aging_score'].values #使用周期伪标签进行训练

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [50]:
# --- XGBoost 回归模型训练 ---
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [51]:
# 在测试集上评估模型性能
y_pred = xgb_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"XGBoost Test MSE: {mse:.4f}, R2: {r2:.4f}")

XGBoost Test MSE: 0.0043, R2: 0.9698


In [56]:
# 预测整个数据集的衰老得分

df_all['predicted_age'] = xgb_model.predict(X)

# 按周期和细胞ID分组并计算平均预测衰老概率
cell_pred = df_all.groupby(['cell_id'])['predicted_age'].mean().reset_index()
cell_pred.to_csv('cell_aging_predictions.csv', index=False)

# --- 可视化：衰老概率分布 ---
plt.figure(figsize=(8,6))
sns.histplot(df_all['predicted_age'],  kde=True, bins=30)
plt.title('age_distribution_overall.png')
plt.xlabel('pred_score')
plt.ylabel('sample_numbers')
plt.tight_layout()
plt.savefig('age_distribution_overall.png')
plt.close()

plt.figure(figsize=(8,6))
sns.histplot(data=df_all, x='predicted_age', hue='cycle',
             bins=30, element='step', stat='density')
plt.title('age_distribution_by_cycle')
plt.xlabel('pred_score')
plt.ylabel('dense')
plt.tight_layout()
plt.savefig('age_distribution_by_cycle.png')
plt.close()

df_all.items

<bound method DataFrame.items of       height  roughness  length  wideth  adhesion  elastic_modulus cycle  \
0      3.239     131.49  108.00   39.72     448.5            5.894    P6   
1      3.315     167.50  102.40   48.05     510.9            2.375    P6   
2      3.112     156.20   98.34   36.55     478.1            5.399    P4   
3      2.231     146.90   72.35   22.23     511.8            3.435    P6   
4      2.386     158.10   98.13   36.54     574.8            5.088    P8   
...      ...        ...     ...     ...       ...              ...   ...   
7995   2.388     256.80  107.70   41.76     535.8            4.895    P8   
7996   2.384      76.40   99.63   29.33     590.0            3.992    P8   
7997   2.235     207.05   82.59   19.24     493.3            2.683    P4   
7998   2.550     120.70   99.69   31.05     617.7            7.460   P10   
7999   2.640      81.70   85.20   29.29     827.8            7.110   P10   

      aging_score  cell_id  predicted_age  
0         

In [53]:
csv_filename_pred = f"pred.csv"
csv_path_pred = os.path.join(data_folder, csv_filename_pred)
# 保存为CSV

df_all.to_csv(csv_path_pred, index=False, encoding='utf-8-sig')
print(f"✅ 已转换子表: pred.csv")


✅ 已转换子表: pred.csv
